# Generador de SFX de Skills con IA (AudioLDM2 + ffmpeg → .ogg)

Genera gratuitamente los efectos de sonido de **magia, fuego, hielo, arcos, escudos, veneno, sismo y nova** usando GPU T4 en Google Colab y actualiza `sfx/manifest.json`, conectado directamente a tu carpeta `Creador-de-mundos-Rpg-Colab` en Google Drive.

**Recomendación:** En Colab activa `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.

In [ ]:
# 1. Verificar GPU e instalar dependencias (diffusers, transformers, accelerate, scipy, ffmpeg)
!nvidia-smi
!pip install -q diffusers transformers accelerate scipy
!apt-get -qq update && apt-get -qq install -y ffmpeg

In [ ]:
# 2. Conectar automáticamente con Google Drive (Creador-de-mundos-Rpg-Colab)
from google.colab import drive, files
import os, shutil

drive.mount('/content/drive')
WORK_DIR = '/content/drive/MyDrive/Creador-de-mundos-Rpg-Colab'
if os.path.isdir(WORK_DIR):
    os.chdir(WORK_DIR)
    print('Carpeta de Drive conectada:', os.getcwd())
else:
    print('No se encontró', WORK_DIR, '— creando carpeta local y pidiendo subida manual...')
    os.makedirs('sfx', exist_ok=True)
    os.makedirs('tools', exist_ok=True)
    uploaded = files.upload()
    if os.path.exists('prompts.json') and not os.path.exists('sfx/prompts.json'):
        shutil.move('prompts.json', 'sfx/prompts.json')
    if os.path.exists('generar-sfx-ia.py') and not os.path.exists('tools/generar-sfx-ia.py'):
        shutil.move('generar-sfx-ia.py', 'tools/generar-sfx-ia.py')

assert os.path.exists('sfx/prompts.json'), 'Falta sfx/prompts.json'
assert os.path.exists('tools/generar-sfx-ia.py'), 'Falta tools/generar-sfx-ia.py'
print('Archivos listos en', os.getcwd())

In [ ]:
# 3. Generar los SFX de magia y arcos (y actualizar sfx/manifest.json)
!python tools/generar-sfx-ia.py --categoria magia --variantes 2 --pasos 60
!python tools/generar-sfx-ia.py --categoria arcos --variantes 2 --pasos 60
!python tools/generar-sfx-ia.py --solo torbellino --variantes 2 --pasos 60

In [ ]:
# 4. Guardar sfx_skills_ogg.zip en Drive (y ofrecer descarga al navegador)
import os, shutil
from google.colab import files

zip_path = shutil.make_archive('sfx_skills_ogg', 'zip', '.', 'sfx')
print('ZIP guardado en Drive:', os.path.abspath(zip_path))
try:
    files.download('sfx_skills_ogg.zip')
except Exception as e:
    print('Descarga directa omitida (ya quedó guardado en tu Google Drive):', e)